# 05C1 — Robust Calibration and Baseline Interpretation Audit

**Build ID:** `05C1_V1_BASELINE_CALIBRATION_INTERPRETATION_AUDIT`

This is a **post-processing-only** notebook for the completed 05C OOF
predictions.

It does not refit a model. It:

- reproduces the saved 05C predictive metrics;
- replaces the warning-prone calibration optimizer with stable Newton/IRLS;
- reports calibration-in-the-large plus calibration slope;
- audits how the three training-weight schemes shift the target prevalence;
- creates clean feature/weighting comparison tables and reliability plots.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05C1_V1_BASELINE_CALIBRATION_INTERPRETATION_AUDIT"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

BASELINE_OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" / "baseline_modeling_v1"
)

PREDICTION_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "baseline_modeling_v1" / "oof_predictions"
)

OOF_SUMMARY_05C_PATH = (
    BASELINE_OUTPUT_ROOT / "OOF_METRIC_SUMMARY_V1.csv"
)

UNIT_DESIGN_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "modeling_design_v1" /
    "unit_outer_fold_and_training_weights_v1.parquet"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "baseline_calibration_audit_v1"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for p in [
    PREDICTION_ROOT,
    OOF_SUMMARY_05C_PATH,
    UNIT_DESIGN_PATH,
]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381

PRED_EPS = 1e-6
MAX_NEWTON_ITER = 100
NEWTON_TOL = 1e-10
RIDGE = 1e-10

print("RUNNING:", NOTEBOOK_BUILD_ID)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

from scipy.special import expit
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

## 2. Validate saved OOF predictions

In [ ]:
EXPECTED_EXPERIMENTS = [
    "TRAINING_PREVALENCE_PRIOR",
    "A_APPROACH_ONLY__incident_equal",
    "B_STATIC_ENV_CORE__incident_equal",
    "C_NEAR_WEATHER_ONLY__incident_equal",
    "D_FULL_CORE__incident_equal",
    "E_FULL_EXTENDED__incident_equal",
    "F_FULL_COVERAGE_STABLE__incident_equal",
    "E_FULL_EXTENDED__row",
    "E_FULL_EXTENDED__incident_class_equal",
]

prediction_paths = {
    p.stem: p
    for p in PREDICTION_ROOT.glob("*.parquet")
}

missing = [
    x for x in EXPECTED_EXPERIMENTS
    if x not in prediction_paths
]

if missing:
    raise RuntimeError(
        f"Missing prediction files: {missing}"
    )

rows = []

for experiment_id in EXPECTED_EXPERIMENTS:
    x = pd.read_parquet(
        prediction_paths[experiment_id],
        columns=[
            "unit_uid",
            "_irwin_norm",
            "hold_label",
            "outer_fold",
            "pred_prob",
        ],
    )

    rows.append({
        "experiment_id": experiment_id,
        "rows": len(x),
        "unique_units": x["unit_uid"].nunique(),
        "unique_incidents": x["_irwin_norm"].nunique(),
        "finite_predictions": bool(
            np.isfinite(
                x["pred_prob"].to_numpy(float)
            ).all()
        ),
        "incident_one_outer_fold": bool(
            (
                x.groupby("_irwin_norm")[
                    "outer_fold"
                ].nunique()
                == 1
            ).all()
        ),
    })

prediction_audit = pd.DataFrame(rows)

display(prediction_audit)

assert (prediction_audit["rows"] == EXPECTED_UNITS).all()
assert (
    prediction_audit["unique_units"]
    == EXPECTED_UNITS
).all()
assert (
    prediction_audit["unique_incidents"]
    == EXPECTED_INCIDENTS
).all()
assert prediction_audit["finite_predictions"].all()
assert prediction_audit["incident_one_outer_fold"].all()

prediction_audit.to_csv(
    OUTPUT_ROOT / "OOF_PREDICTION_INPUT_AUDIT_V1.csv",
    index=False,
)

print("OOF prediction contract: PASS")

## 3. Stable calibration estimators

For predicted probability \(p\), define \(z=\mathrm{logit}(p)\).

**Calibration-in-the-large** fits `logit(Y) = a + offset(z)`. Ideal: `a=0`.

**Calibration slope** fits `logit(Y) = a + b*z`. Ideal: `a=0, b=1`.

The implementation uses analytic Newton/IRLS updates rather than the BFGS
optimizer that produced precision-loss warnings in 05C.

In [ ]:
def clip_prob(p):
    return np.clip(
        np.asarray(p, dtype=float),
        PRED_EPS,
        1.0 - PRED_EPS,
    )


def logit_prob(p):
    p = clip_prob(p)
    return np.log(
        p / (1.0 - p)
    )


def calibration_in_the_large(y, p):
    y = np.asarray(y, dtype=float)
    z = logit_prob(p)

    a = 0.0
    converged = False

    for iteration in range(MAX_NEWTON_ITER):
        mu = expit(a + z)

        score = np.sum(y - mu)

        information = (
            np.sum(
                mu * (1.0 - mu)
            )
            + RIDGE
        )

        step = score / information
        a = a + step

        if abs(step) < NEWTON_TOL:
            converged = True
            break

    return {
        "calibration_in_the_large": float(a),
        "citl_converged": bool(converged),
        "citl_iterations": int(iteration + 1),
    }


def calibration_joint_intercept_slope(y, p):
    y = np.asarray(y, dtype=float)
    z = logit_prob(p)

    X = np.column_stack([
        np.ones_like(z),
        z,
    ])

    beta = np.array(
        [0.0, 1.0],
        dtype=float,
    )

    converged = False
    cond = np.nan

    for iteration in range(MAX_NEWTON_ITER):
        eta = X @ beta
        mu = expit(eta)

        w = mu * (1.0 - mu)

        score = X.T @ (y - mu)

        information = (
            X.T @ (X * w[:, None])
            + RIDGE * np.eye(2)
        )

        cond = float(
            np.linalg.cond(information)
        )

        step = np.linalg.solve(
            information,
            score,
        )

        beta = beta + step

        if np.max(
            np.abs(step)
        ) < NEWTON_TOL:
            converged = True
            break

    return {
        "calibration_joint_intercept": float(beta[0]),
        "calibration_slope": float(beta[1]),
        "joint_converged": bool(converged),
        "joint_iterations": int(iteration + 1),
        "calibration_information_condition_number": cond,
    }


# Synthetic implementation check.
rng = np.random.default_rng(20260825)

p_syn = rng.uniform(
    0.05,
    0.95,
    size=200_000,
)

y_syn = rng.binomial(1, p_syn)

s1 = calibration_in_the_large(
    y_syn,
    p_syn,
)

s2 = calibration_joint_intercept_slope(
    y_syn,
    p_syn,
)

print("Synthetic calibration check:", s1, s2)

assert abs(
    s1["calibration_in_the_large"]
) < 0.03

assert abs(
    s2["calibration_joint_intercept"]
) < 0.05

assert abs(
    s2["calibration_slope"] - 1.0
) < 0.05

## 4. Predictive metric helpers

In [ ]:
def binary_log_loss_vector(y, p):
    y = np.asarray(y, dtype=float)
    p = clip_prob(p)

    return -(
        y * np.log(p)
        + (1.0 - y) * np.log(1.0 - p)
    )


def per_incident_metrics(frame):
    rows = []

    for incident, g in frame.groupby(
        "_irwin_norm",
        sort=False,
    ):
        y = g["hold_label"].to_numpy(int)
        p = g["pred_prob"].to_numpy(float)

        mixed = (
            np.unique(y).size == 2
        )

        rows.append({
            "_irwin_norm": incident,
            "units": len(g),
            "brier": float(
                np.mean((p - y) ** 2)
            ),
            "log_loss": float(
                np.mean(
                    binary_log_loss_vector(y, p)
                )
            ),
            "mixed_class": mixed,
            "pr_auc": (
                float(
                    average_precision_score(y, p)
                )
                if mixed
                else np.nan
            ),
            "roc_auc": (
                float(
                    roc_auc_score(y, p)
                )
                if mixed
                else np.nan
            ),
        })

    return pd.DataFrame(rows)


def qcut_ece(frame, bins=10):
    x = frame[
        ["hold_label", "pred_prob"]
    ].copy()

    try:
        x["bin"] = pd.qcut(
            x["pred_prob"],
            q=bins,
            duplicates="drop",
        )
    except ValueError:
        x["bin"] = pd.cut(
            x["pred_prob"],
            bins=bins,
            include_lowest=True,
        )

    rel = (
        x.groupby(
            "bin",
            observed=True,
            as_index=False,
        )
        .agg(
            units=("hold_label", "size"),
            mean_pred=("pred_prob", "mean"),
            observed_rate=("hold_label", "mean"),
        )
    )

    rel["abs_gap"] = (
        rel["mean_pred"]
        - rel["observed_rate"]
    ).abs()

    rel["weight"] = (
        rel["units"] / rel["units"].sum()
    )

    ece = float(
        (
            rel["weight"]
            * rel["abs_gap"]
        ).sum()
    )

    return rel, ece


def fixed_width_reliability(frame, bins=10):
    edges = np.linspace(
        0.0,
        1.0,
        bins + 1,
    )

    x = frame[
        ["hold_label", "pred_prob"]
    ].copy()

    x["bin"] = pd.cut(
        x["pred_prob"],
        bins=edges,
        include_lowest=True,
    )

    rel = (
        x.groupby(
            "bin",
            observed=True,
            as_index=False,
        )
        .agg(
            units=("hold_label", "size"),
            mean_pred=("pred_prob", "mean"),
            observed_rate=("hold_label", "mean"),
        )
    )

    rel["abs_gap"] = (
        rel["mean_pred"]
        - rel["observed_rate"]
    ).abs()

    rel["weight"] = (
        rel["units"] / rel["units"].sum()
    )

    return rel


def summarize(frame):
    y = frame["hold_label"].to_numpy(int)
    p = frame["pred_prob"].to_numpy(float)

    inc = per_incident_metrics(frame)
    mixed = inc["mixed_class"]

    _, ece = qcut_ece(frame)

    return {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "held_rate": float(np.mean(y)),
        "mean_pred": float(np.mean(p)),
        "prediction_bias_mean_pred_minus_observed":
        float(np.mean(p) - np.mean(y)),
        "pooled_pr_auc":
        float(average_precision_score(y, p)),
        "pooled_roc_auc":
        float(roc_auc_score(y, p)),
        "pooled_brier":
        float(brier_score_loss(y, p)),
        "pooled_log_loss":
        float(
            log_loss(
                y,
                clip_prob(p),
                labels=[0, 1],
            )
        ),
        "incident_macro_brier":
        float(inc["brier"].mean()),
        "incident_macro_log_loss":
        float(inc["log_loss"].mean()),
        "mixed_incidents":
        int(mixed.sum()),
        "mixed_incident_macro_pr_auc":
        float(
            inc.loc[mixed, "pr_auc"].mean()
        ),
        "mixed_incident_macro_roc_auc":
        float(
            inc.loc[mixed, "roc_auc"].mean()
        ),
        "ece_qcut": ece,
        **calibration_in_the_large(y, p),
        **calibration_joint_intercept_slope(y, p),
    }, inc

## 5. Recompute metrics and robust calibration

In [ ]:
summary_rows = []
incident_frames = []
reliability_frames = []

for experiment_id in EXPECTED_EXPERIMENTS:
    frame = pd.read_parquet(
        prediction_paths[experiment_id]
    )

    metrics, inc = summarize(frame)
    metrics["experiment_id"] = experiment_id

    summary_rows.append(metrics)

    inc["experiment_id"] = experiment_id
    incident_frames.append(inc)

    rel = fixed_width_reliability(frame)
    rel["experiment_id"] = experiment_id
    reliability_frames.append(rel)

robust_summary = pd.DataFrame(summary_rows)

per_incident = pd.concat(
    incident_frames,
    ignore_index=True,
)

fixed_reliability = pd.concat(
    reliability_frames,
    ignore_index=True,
)

display(
    robust_summary.sort_values(
        "pooled_pr_auc",
        ascending=False,
    )
)

assert robust_summary[
    "citl_converged"
].all()

assert robust_summary[
    "joint_converged"
].all()

robust_summary.to_csv(
    OUTPUT_ROOT /
    "ROBUST_OOF_METRICS_AND_CALIBRATION_V1.csv",
    index=False,
)

per_incident.to_parquet(
    OUTPUT_ROOT /
    "ROBUST_PER_INCIDENT_METRICS_V1.parquet",
    index=False,
)

fixed_reliability.to_csv(
    OUTPUT_ROOT /
    "FIXED_WIDTH_RELIABILITY_BINS_V1.csv",
    index=False,
)

## 6. Prove 05C predictive metrics are unchanged

In [ ]:
old = pd.read_csv(
    OOF_SUMMARY_05C_PATH
)

COMPARE_METRICS = [
    "pooled_pr_auc",
    "pooled_roc_auc",
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
]

new = robust_summary[
    [
        "experiment_id",
        *COMPARE_METRICS,
        "ece_qcut",
    ]
].rename(
    columns={
        "ece_qcut": "ece",
    }
)

cmp = old[
    [
        "experiment_id",
        *COMPARE_METRICS,
        "ece",
    ]
].merge(
    new,
    on="experiment_id",
    suffixes=("_05c", "_05c1"),
    validate="one_to_one",
)

diff_rows = []

for _, row in cmp.iterrows():
    for metric in [
        *COMPARE_METRICS,
        "ece",
    ]:
        diff_rows.append({
            "experiment_id": row["experiment_id"],
            "metric": metric,
            "05c": row[f"{metric}_05c"],
            "05c1": row[f"{metric}_05c1"],
            "abs_difference": abs(
                row[f"{metric}_05c"]
                - row[f"{metric}_05c1"]
            ),
        })

metric_reproduction = pd.DataFrame(
    diff_rows
)

MAX_REPRO_DIFF = float(
    metric_reproduction[
        "abs_difference"
    ].max()
)

display(
    metric_reproduction.sort_values(
        "abs_difference",
        ascending=False,
    ).head(30)
)

print(
    "Max reproduction difference:",
    MAX_REPRO_DIFF,
)

assert MAX_REPRO_DIFF < 1e-10

metric_reproduction.to_csv(
    OUTPUT_ROOT /
    "05C_METRIC_REPRODUCTION_AUDIT_V1.csv",
    index=False,
)

print("05C predictive metrics unchanged: PASS")

## 7. Weighting changes the probability estimand

Incident-equal and incident-class-equal fitting deliberately change the
empirical distribution seen by the loss function. Their raw probabilities
therefore need not have the same mean as the unweighted unit population.

This cell quantifies that shift.

In [ ]:
unit_design = pd.read_parquet(
    UNIT_DESIGN_PATH,
    columns=[
        "hold_label",
        "weight_row_raw",
        "weight_incident_equal_raw",
        "weight_incident_class_equal_raw",
    ],
)

WEIGHT_MAP = {
    "row": "weight_row_raw",
    "incident_equal":
    "weight_incident_equal_raw",
    "incident_class_equal":
    "weight_incident_class_equal_raw",
}

FULL_MODEL_ID = {
    "row":
    "E_FULL_EXTENDED__row",
    "incident_equal":
    "E_FULL_EXTENDED__incident_equal",
    "incident_class_equal":
    "E_FULL_EXTENDED__incident_class_equal",
}

lookup = robust_summary.set_index(
    "experiment_id"
)

rows = []

y = unit_design[
    "hold_label"
].to_numpy(float)

for scheme, weight_col in WEIGHT_MAP.items():
    w = unit_design[
        weight_col
    ].to_numpy(float)

    exp = FULL_MODEL_ID[scheme]

    rows.append({
        "weight_scheme": scheme,
        "unweighted_unit_prevalence":
        float(y.mean()),
        "training_raw_weighted_prevalence":
        float(
            np.average(y, weights=w)
        ),
        "full_extended_oof_mean_prediction":
        float(
            lookup.loc[exp, "mean_pred"]
        ),
        "oof_mean_minus_unweighted_prevalence":
        float(
            lookup.loc[exp, "mean_pred"]
            - y.mean()
        ),
    })

weight_prevalence = pd.DataFrame(rows)

display(weight_prevalence)

weight_prevalence.to_csv(
    OUTPUT_ROOT /
    "WEIGHTING_TARGET_PREVALENCE_AUDIT_V1.csv",
    index=False,
)

## 8. Clean ablation tables and pre-specified deltas

In [ ]:
FEATURE_IDS = [
    "A_APPROACH_ONLY__incident_equal",
    "B_STATIC_ENV_CORE__incident_equal",
    "C_NEAR_WEATHER_ONLY__incident_equal",
    "D_FULL_CORE__incident_equal",
    "E_FULL_EXTENDED__incident_equal",
    "F_FULL_COVERAGE_STABLE__incident_equal",
]

WEIGHT_IDS = [
    "E_FULL_EXTENDED__row",
    "E_FULL_EXTENDED__incident_equal",
    "E_FULL_EXTENDED__incident_class_equal",
]

DISPLAY_COLUMNS = [
    "experiment_id",
    "pooled_pr_auc",
    "pooled_roc_auc",
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
    "mean_pred",
    "prediction_bias_mean_pred_minus_observed",
    "calibration_in_the_large",
    "calibration_slope",
    "ece_qcut",
]

feature_comparison = robust_summary[
    robust_summary["experiment_id"].isin(
        FEATURE_IDS
    )
][DISPLAY_COLUMNS].copy()

weight_comparison = robust_summary[
    robust_summary["experiment_id"].isin(
        WEIGHT_IDS
    )
][DISPLAY_COLUMNS].copy()

display(
    feature_comparison.sort_values(
        "pooled_pr_auc",
        ascending=False,
    )
)

display(
    weight_comparison.sort_values(
        "pooled_pr_auc",
        ascending=False,
    )
)

feature_comparison.to_csv(
    OUTPUT_ROOT /
    "FEATURE_ABLATION_ROBUST_SUMMARY_V1.csv",
    index=False,
)

weight_comparison.to_csv(
    OUTPUT_ROOT /
    "WEIGHT_ABLATION_ROBUST_SUMMARY_V1.csv",
    index=False,
)

PAIR_COMPARISONS = [
    (
        "extended_vs_core",
        "E_FULL_EXTENDED__incident_equal",
        "D_FULL_CORE__incident_equal",
    ),
    (
        "full_vs_approach",
        "E_FULL_EXTENDED__incident_equal",
        "A_APPROACH_ONLY__incident_equal",
    ),
    (
        "full_vs_static",
        "E_FULL_EXTENDED__incident_equal",
        "B_STATIC_ENV_CORE__incident_equal",
    ),
    (
        "coverage_stable_vs_extended",
        "F_FULL_COVERAGE_STABLE__incident_equal",
        "E_FULL_EXTENDED__incident_equal",
    ),
    (
        "row_vs_incident_equal",
        "E_FULL_EXTENDED__row",
        "E_FULL_EXTENDED__incident_equal",
    ),
    (
        "incident_class_equal_vs_incident_equal",
        "E_FULL_EXTENDED__incident_class_equal",
        "E_FULL_EXTENDED__incident_equal",
    ),
]

DELTA_METRICS = [
    "pooled_pr_auc",
    "pooled_roc_auc",
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
    "ece_qcut",
]

delta_rows = []

for label, left, right in PAIR_COMPARISONS:
    row = {
        "comparison": label,
        "left_experiment": left,
        "right_experiment": right,
    }

    for metric in DELTA_METRICS:
        row[
            f"delta_{metric}"
        ] = float(
            lookup.loc[left, metric]
            - lookup.loc[right, metric]
        )

    delta_rows.append(row)

scientific_deltas = pd.DataFrame(
    delta_rows
)

display(scientific_deltas)

scientific_deltas.to_csv(
    OUTPUT_ROOT /
    "PRESPECIFIED_BASELINE_DELTAS_V1.csv",
    index=False,
)

## 9. Reliability plots

In [ ]:
PLOT_EXPERIMENTS = [
    "TRAINING_PREVALENCE_PRIOR",
    "A_APPROACH_ONLY__incident_equal",
    "B_STATIC_ENV_CORE__incident_equal",
    "C_NEAR_WEATHER_ONLY__incident_equal",
    "D_FULL_CORE__incident_equal",
    "E_FULL_EXTENDED__incident_equal",
    "E_FULL_EXTENDED__row",
    "E_FULL_EXTENDED__incident_class_equal",
]

for experiment_id in PLOT_EXPERIMENTS:
    rel = fixed_reliability[
        fixed_reliability[
            "experiment_id"
        ] == experiment_id
    ]

    fig, ax = plt.subplots(
        figsize=(6, 6)
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
    )

    ax.plot(
        rel["mean_pred"],
        rel["observed_rate"],
        marker="o",
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(
        "Mean predicted hold probability"
    )
    ax.set_ylabel(
        "Observed hold rate"
    )
    ax.set_title(experiment_id)
    ax.grid(True, alpha=0.25)

    fig.tight_layout()

    fig.savefig(
        OUTPUT_ROOT /
        f"RELIABILITY_{experiment_id}_V1.png",
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(fig)

print(
    "Reliability plots saved:",
    len(PLOT_EXPERIMENTS),
)

## 10. Final checks and report

In [ ]:
checks = pd.DataFrame([
    {
        "check":
        "all_nine_prediction_sets_present",
        "pass":
        len(prediction_audit) == 9,
    },
    {
        "check":
        "all_prediction_sets_304143_rows",
        "pass":
        bool(
            (
                prediction_audit["rows"]
                == EXPECTED_UNITS
            ).all()
        ),
    },
    {
        "check":
        "all_05c_predictive_metrics_reproduced",
        "pass":
        MAX_REPRO_DIFF < 1e-10,
    },
    {
        "check":
        "all_citl_fits_converged",
        "pass":
        bool(
            robust_summary[
                "citl_converged"
            ].all()
        ),
    },
    {
        "check":
        "all_joint_calibration_fits_converged",
        "pass":
        bool(
            robust_summary[
                "joint_converged"
            ].all()
        ),
    },
    {
        "check":
        "weighting_prevalence_audit_complete",
        "pass":
        len(weight_prevalence) == 3,
    },
    {
        "check": "no_model_refit",
        "pass": True,
    },
    {
        "check": "no_hyperparameter_tuning",
        "pass": True,
    },
    {
        "check": "no_threshold_selection",
        "pass": True,
    },
])

display(checks)

AUDIT_PASS = bool(
    checks["pass"].all()
)

print()
print(
    "05C1 CALIBRATION / INTERPRETATION AUDIT:",
    (
        "PASS / BASELINE OOF RESULTS READY FOR SCIENTIFIC INTERPRETATION"
        if AUDIT_PASS
        else "REVIEW REQUIRED"
    ),
)

checks.to_csv(
    OUTPUT_ROOT /
    "FINAL_CALIBRATION_AUDIT_CHECKS_V1.csv",
    index=False,
)

def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

nonprior = robust_summary[
    robust_summary[
        "experiment_id"
    ] != "TRAINING_PREVALENCE_PRIOR"
]

best_pr = nonprior.sort_values(
    "pooled_pr_auc",
    ascending=False,
).iloc[0]

best_brier = nonprior.sort_values(
    "pooled_brier",
    ascending=True,
).iloc[0]

best_macro_roc = nonprior.sort_values(
    "mixed_incident_macro_roc_auc",
    ascending=False,
).iloc[0]

report = f"""# 05C1 Robust Calibration and Baseline Interpretation Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Robust OOF summary

{tomd(robust_summary.sort_values('pooled_pr_auc', ascending=False))}

## Weighting prevalence audit

{tomd(weight_prevalence)}

## Feature ablation

{tomd(feature_comparison)}

## Weight ablation

{tomd(weight_comparison)}

## Pre-specified deltas

{tomd(scientific_deltas)}

## Best descriptive baseline results

Best pooled PR-AUC:
`{best_pr['experiment_id']}` = `{best_pr['pooled_pr_auc']:.6f}`

Best pooled Brier:
`{best_brier['experiment_id']}` = `{best_brier['pooled_brier']:.6f}`

Best mixed-incident macro ROC-AUC:
`{best_macro_roc['experiment_id']}` =
`{best_macro_roc['mixed_incident_macro_roc_auc']:.6f}`

## Final checks

{tomd(checks)}

## Decision

`{"PASS / BASELINE OOF RESULTS READY FOR SCIENTIFIC INTERPRETATION" if AUDIT_PASS else "REVIEW REQUIRED"}`

No model was refitted in 05C1.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "BASELINE_CALIBRATION_INTERPRETATION_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print("Saved:", REPORT_PATH)

## 11. Output inventory

In [ ]:
print("=" * 100)
print("05C1 OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob("*")
):
    if p.is_file():
        print(
            f"{p.name:<105}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)